# Bakery Market Basket Analysis Using KDD

## Phase 1 — Selection — Initial Draft

### Objective and intended use
Discover products frequently purchased together and use association
rules to demonstrate suggestions for an existing basket.

The results describe historical co-purchases. They do not establish
purchase order, causation, or increased sales from recommendations.

### Data source and selected records
Source: https://www.kaggle.com/datasets/akashdeepkuila/bakery

I use Bakery.csv, containing 20,507 item rows and five columns:
TransactionNo, Items, DateTime, Daypart, and DayType.

The file contains 9,465 distinct transaction numbers and 94 item
names. I retain all records initially because the dataset is
manageable in Colab.

### Select relevant fields
TransactionNo identifies the basket and Items identifies its contents.
These two fields will support association-rule mining.

DateTime will be retained for checking transaction consistency.
Daypart and DayType are excluded because this project examines
overall co-purchases rather than time-specific patterns.

The working assumption is that each transaction number identifies
one purchase event. Timestamp consistency will be checked during
preprocessing.

### Planned knowledge discovery
After cleaning, rows will be grouped into baskets containing unique
items. Apriori will identify frequent itemsets, and association rules
will be evaluated using support, confidence, and lift. Thresholds
will be documented before mining.

The deliverable will include interpretable rules, manual verification
of their metrics, and a saved-rule recommendation demonstration.
No supervised target variable is required.

### Scope and limitations
The analysis describes the available historical baskets. It does
not establish accuracy for future purchases or other businesses.

Repeated item entries may represent multiple units. Their presence
alone does not establish an error. For this presence-based analysis,
each item will eventually count once per basket.

In [1]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv(io.BytesIO(uploaded["Bakery.csv"]))

required_columns = ["TransactionNo", "Items", "DateTime"]
assert set(required_columns).issubset(df.columns)

selected_df = df[required_columns].copy()

print("Original shape:", df.shape)
print("Selected columns:", selected_df.columns.tolist())
print("Distinct transaction numbers:", df["TransactionNo"].nunique())
print("Distinct item names:", df["Items"].nunique())
print("Missing values:", df.isna().sum().sum())
print(
    "Blank text values:",
    df.select_dtypes(include="object")
      .apply(lambda col: col.str.strip().eq("").sum())
      .sum()
)
print("Repeated full rows:", df.duplicated().sum())

assert len(selected_df) == len(df)
assert selected_df.index.equals(df.index)

display(selected_df.head())
print("Selection preserved all rows and their original order.")

Saving Bakery.csv to Bakery.csv
Original shape: (20507, 5)
Selected columns: ['TransactionNo', 'Items', 'DateTime']
Distinct transaction numbers: 9465
Distinct item names: 94
Missing values: 0
Blank text values: 0
Repeated full rows: 1620


,TransactionNo,Items,DateTime
0,1,Bread,2016-10-30 09:58:11
1,2,Scandinavian,2016-10-30 10:05:34
2,2,Scandinavian,2016-10-30 10:05:34
3,3,Hot chocolate,2016-10-30 10:07:57
4,3,Jam,2016-10-30 10:07:57


Selection preserved all rows and their original order.


### Phase 1 — Selection — Results

The loaded file contains 20,507 item rows, five columns,
9,465 distinct transaction numbers, and 94 distinct item names.

TransactionNo, Items, and DateTime were selected. All rows and
their original order were preserved.

Initial checks found zero pandas-recognized missing values,
zero blank text values, and 1,620 repeated full rows.

Repeated rows have not been removed. They may represent multiple
units, but their meaning is unverified. During preprocessing,
repeated transaction-item pairs will be collapsed for the planned
presence-based basket analysis.

Timestamp consistency, cleaning, basket construction, and rule
mining remain pending.

## Phase 1 — Selection — Revision 1

The initial draft, selection code, and results are retained unchanged
following Round 1, which identified no essential corrections.

All 20,507 item rows remain selected, using TransactionNo, Items,
and DateTime. The 1,620 repeated full rows remain documented
without assuming they are errors or verified quantities.

Treating each transaction number as one purchase event remains
an assumption pending timestamp-consistency checks.

No additional execution or modifications were performed.
Preprocessing, basket construction, rule mining, and evaluation
remain pending.

##Phase 1 — Selection — Critique Round 2

## KDD Phase 1 — Selection: Second Review (Revision 1)

Approved — No essential corrections

Assessment: Revision 1 remains adequately documented and methodologically sound.

- Selection: All 20,507 rows and the three selected fields remain appropriately justified.
- Repeated rows: The 1,620 repeated full rows are correctly documented without assuming errors or verified quantities.
- Assumptions: Transaction identifiers are treated as purchase events provisionally, pending timestamp-consistency checks.
- Verification: Previously executed results are retained accurately. No additional execution is claimed.
- Pending work: Preprocessing, basket construction, rule mining, and evaluation remain clearly identified.

### Final Verdict

Phase 1 — Selection is approved and complete.

All six review criteria remain satisfied. No essential issues remain, and no revisions or additional work are required for this phase.

Ready to proceed to Phase 2 — Preprocessing.

## Phase 1 — Selection — Final Revision

Revision 1 is adopted as final. Round 2 identified no essential
corrections.

## Phase 2 — Preprocessing — Initial Draft

### Validate required values
I will check the selected fields for missing and blank values.
Transaction numbers must be positive integers, and DateTime
must parse successfully. Invalid values will stop processing
rather than be silently removed or imputed.

### Check transaction consistency
I will verify that each transaction number has exactly one
timestamp. This supports consistent grouping within the file,
but does not independently prove that the identifier represents
a real purchase event.

### Prepare item labels
Leading and trailing whitespace will be removed from item names.
Capitalization and internal spelling will otherwise be preserved
to avoid merging potentially distinct products.

### Handle repeated items
Repeated transaction-item pairs will be collapsed so each item
counts once per basket. This is a presence-based representation
decision, not a claim that repeated entries were recording errors.

### Verify the cleaned data
Checks will confirm that transaction-item pairs are unique and
that the set of transaction numbers is preserved. Basket grouping
and binary encoding remain part of the Transformation phase.

In [4]:
import numpy as np

clean_df = selected_df.copy()

assert not clean_df.isna().any().any(), "Missing values need review."

for column in ["TransactionNo", "Items", "DateTime"]:
    assert not clean_df[column].astype(str).str.strip().eq("").any(), (
        f"Blank values in {column} need review."
    )

transaction_numbers = pd.to_numeric(
    clean_df["TransactionNo"], errors="raise"
)

assert np.isfinite(transaction_numbers.to_numpy()).all()
assert transaction_numbers.gt(0).all()
assert transaction_numbers.mod(1).eq(0).all()

clean_df["TransactionNo"] = transaction_numbers.astype("int64")
clean_df["DateTime"] = pd.to_datetime(
    clean_df["DateTime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise"
)

timestamp_counts = clean_df.groupby("TransactionNo")["DateTime"].nunique()
inconsistent_transactions = int(timestamp_counts.ne(1).sum())

print("Transactions with inconsistent timestamps:",
      inconsistent_transactions)
assert inconsistent_transactions == 0, "Review transaction grouping."

original_items = clean_df["Items"].copy()
clean_df["Items"] = clean_df["Items"].str.strip()

print("Item labels changed by trimming:",
      int(original_items.ne(clean_df["Items"]).sum()))

transaction_ids_before = set(clean_df["TransactionNo"])
rows_before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["TransactionNo", "Items"]
).copy()

assert not clean_df.duplicated(["TransactionNo", "Items"]).any()
assert set(clean_df["TransactionNo"]) == transaction_ids_before
assert not clean_df.isna().any().any()
assert clean_df["Items"].str.len().gt(0).all()

print("Rows before collapsing repeated items:", rows_before)
print("Repeated transaction-item rows collapsed:",
      rows_before - len(clean_df))
print("Cleaned transaction-item rows:", len(clean_df))
print("Transactions retained:", clean_df["TransactionNo"].nunique())
print("Distinct item names:", clean_df["Items"].nunique())
print("Observed timestamp range:",
      clean_df["DateTime"].min(), "to", clean_df["DateTime"].max())
print("Uniqueness and transaction-preservation checks passed.")

display(clean_df.head())

Transactions with inconsistent timestamps: 0
Item labels changed by trimming: 21
Rows before collapsing repeated items: 20507
Repeated transaction-item rows collapsed: 1620
Cleaned transaction-item rows: 18887
Transactions retained: 9465
Distinct item names: 94
Observed timestamp range: 2016-01-11 07:51:20 to 2017-12-03 16:28:00
Uniqueness and transaction-preservation checks passed.


,TransactionNo,Items,DateTime
0,1,Bread,2016-10-30 09:58:11
1,2,Scandinavian,2016-10-30 10:05:34
3,3,Hot chocolate,2016-10-30 10:07:57
4,3,Jam,2016-10-30 10:07:57
5,3,Cookies,2016-10-30 10:07:57


### Phase 2 — Preprocessing — Results

All required-value, transaction-number, and timestamp-parsing
checks passed. Every transaction number has exactly one timestamp.
This supports consistent grouping within the file, but does not
independently verify the real-world meaning of each identifier.

Whitespace trimming changed 21 item entries. Capitalization and
internal spelling were preserved. There remain 94 distinct item names.

Collapsing 1,620 repeated transaction-item rows reduced the data
from 20,507 to 18,887 rows. All 9,465 transaction numbers were
preserved, and transaction-item uniqueness checks passed.

This collapse represents item presence within baskets. It does not
establish whether repeated entries were quantities or recording errors.

Parsed timestamps range from 2016-01-11 07:51:20 to
2017-12-03 16:28:00. This is the observed range in the supplied file;
the original dates have not been independently verified. Dates will
not determine the overall co-purchase analysis.

Basket construction, binary encoding, and rule mining remain pending.

##Phase 2 — Preprocessing — Critique Round 1

## KDD Phase 2 — Preprocessing: Expert Review

Approved — No essential corrections

Overall assessment: Phase 2 is methodologically sound, adequately documented, and supported by the executed code and reported results.

| Review criterion                   | Status     | Assessment                                                                                                                             |
| ---------------------------------- | ---------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Required-value validation       | &#x20;Pass | Missing values, blanks, positive integer transaction numbers, and timestamp parsing are validated.                                     |
| 2. Timestamp consistency           | &#x20;Pass | All transactions have one timestamp. The distinction between internal consistency and real-world verification is correctly maintained. |
| 3. Item-label trimming             | &#x20;Pass | Trimming affected 21 entries. Capitalization and internal spelling are preserved.                                                      |
| 4. Repeated transaction-item pairs | &#x20;Pass | Collapsing 1,620 repeated pairs is appropriate for presence-based association analysis without assuming recording errors.              |
| 5. Preservation and limitations    | &#x20;Pass | All 9,465 transactions are retained, transaction-item uniqueness is verified, and remaining limitations are documented.                |

### Verified results

The reported reduction from 20,507 to 18,887 rows is consistent with removing 1,620 repeated transaction-item pairs. The 94 distinct items, transaction preservation, and parsed timestamp range are accurately documented.

### Assumptions and pending work

Transaction identifiers are internally consistent but not independently verified as real purchase events. The meaning of repeated entries and the historical timestamps remains unverified.

Basket construction, binary encoding, and association-rule mining are appropriately deferred.

### Essential corrections

None identified.

### Final verdict

Phase 2 — Preprocessing is approved as written.

All five review criteria are satisfied. No essential issues remain, and no code changes or additional analyses are required.

Ready to proceed to Phase 3 — Transformation.

### Phase 2 — Preprocessing — Revision 1

Round 1 identified no essential corrections. The existing preprocessing
code, findings, and limitations are retained unchanged.

The executed checks confirmed 18,887 unique transaction-item pairs,
9,465 retained transactions, and 94 distinct items. Trimming affected
21 item entries, and 1,620 repeated transaction-item rows were collapsed
for presence-based analysis.

Timestamp consistency supports the grouping procedure but does not
independently verify real purchase events or the accuracy of historical
dates. Repeated entries are not assumed to be errors.

No new code was executed for this revision. Basket construction,
binary encoding, and association-rule mining remain pending.

##Phase 2 — Preprocessing — Critique Round 2

## KDD Phase 2 — Preprocessing: Second Review (Revision 1)

Approved — No essential corrections

Overall assessment: Revision 1 remains adequately documented, methodologically sound, and consistent with the previously approved code and execution results.

| Review criterion                   | Status     | Assessment                                                                                                     |
| ---------------------------------- | ---------- | -------------------------------------------------------------------------------------------------------------- |
| 1. Required-value validation       | &#x20;Pass | Missing values, blanks, positive integer transaction numbers, and timestamp parsing were validated in Round 1. |
| 2. Timestamp consistency           | &#x20;Pass | Consistency was verified without claiming independent real-world validation.                                   |
| 3. Item-label trimming             | &#x20;Pass | The 21 trimmed entries are accurately documented.                                                              |
| 4. Repeated transaction-item pairs | &#x20;Pass | All 1,620 repeated pairs were appropriately collapsed for presence-based analysis.                             |
| 5. Preservation and limitations    | &#x20;Pass | All 9,465 transactions were preserved, 18,887 unique pairs remain, and limitations are explicit.               |

### Essential corrections

None identified.

No new execution is claimed, and the revision accurately retains the previously verified findings.

### Final verdict

Phase 2 — Preprocessing is approved and complete.

All five review criteria remain satisfied. No essential corrections, code changes, or additional analyses are required.

Ready to proceed to Phase 3 — Transformation.

## Phase 2 — Preprocessing — Final Revision

Revision 1 is adopted following the second review, which identified
no essential corrections. No additional code changes or execution
were required.

## Phase 3 — Transformation — Initial Draft

### Construct baskets
Group the cleaned item records by TransactionNo. Each transaction
represents one basket containing its distinct item names.

Retain all transactions, including single-item baskets, so support
will use the complete historical transaction population.

### Encode item presence
Create a Boolean transaction-by-item matrix. Rows represent
transactions, columns represent item names, and each value indicates
whether that item appears in the transaction.

True means present and False means absent. This representation
does not measure quantities, purchase order, or spending.

TransactionNo is retained as the row identifier. DateTime is not
included as a mining feature because the objective concerns overall
historical co-purchase patterns.

### Verify the transformation
Check that all transactions and items are represented, every basket
contains at least one item, and the total number of True entries
matches the cleaned transaction-item count.

Reconstruct each basket from the encoded matrix and compare it with
the cleaned records to verify that item membership is preserved.

Association-rule mining and evaluation remain pending.

In [5]:
baskets = (
    clean_df.groupby("TransactionNo", sort=True)["Items"]
    .agg(lambda items: frozenset(items))
)

basket_matrix = pd.crosstab(
    clean_df["TransactionNo"],
    clean_df["Items"]
).gt(0)

basket_matrix = basket_matrix.sort_index().sort_index(axis=1)

assert basket_matrix.index.is_unique
assert basket_matrix.columns.is_unique
assert set(basket_matrix.index) == set(clean_df["TransactionNo"])
assert set(basket_matrix.columns) == set(clean_df["Items"])
assert basket_matrix.dtypes.eq(bool).all()
assert not basket_matrix.isna().any().any()
assert basket_matrix.any(axis=1).all()

presence_count = int(basket_matrix.to_numpy().sum())
assert presence_count == len(clean_df)

for transaction_id, row in basket_matrix.iterrows():
    reconstructed = frozenset(basket_matrix.columns[row.to_numpy()])
    assert reconstructed == baskets.loc[transaction_id]

basket_sizes = basket_matrix.sum(axis=1)

print("Encoded matrix shape:", basket_matrix.shape)
print("Total item presences:", presence_count)
print("Single-item baskets:", int(basket_sizes.eq(1).sum()))
print("Minimum basket size:", int(basket_sizes.min()))
print("Maximum basket size:", int(basket_sizes.max()))
print("Mean basket size:", round(float(basket_sizes.mean()), 2))
print("Boolean encoding and basket-reconstruction checks passed.")

display(basket_matrix.iloc[:5, :10])

Encoded matrix shape: (9465, 94)
Total item presences: 18887
Single-item baskets: 3948
Minimum basket size: 1
Maximum basket size: 10
Mean basket size: 2.0
Boolean encoding and basket-reconstruction checks passed.


Items,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket
TransactionNo,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False


### Phase 3 — Transformation — Results

The encoded matrix contains 9,465 transactions and 94 item columns.
Its 18,887 True values match the cleaned transaction-item count.

All transactions were retained, including 3,948 single-item baskets.
Basket sizes range from 1 to 10 distinct items, with a mean of
approximately 2.00.

Boolean-type, nonempty-basket, transaction-preservation, and
item-preservation checks passed. Reconstructing every basket from
the matrix reproduced its original cleaned item membership.

The displayed preview includes only the first 10 item columns.
False values in this subset do not indicate empty transactions.

The matrix represents item presence, not quantities or purchase
sequence. Association-rule mining and evaluation remain pending.

##Phase 3 — Transformation — Critique Round 1

## KDD Phase 3 — Transformation: Expert Review

Approved — No essential corrections

Overall assessment: Phase 3 is methodologically sound, adequately documented, and supported by the executed code and reported results.

| Review criterion                | Status     | Assessment                                                                                                                   |
| ------------------------------- | ---------- | ---------------------------------------------------------------------------------------------------------------------------- |
| 1. Basket construction          | &#x20;Pass | Transactions are correctly grouped by `TransactionNo`, with distinct items represented in each basket.                       |
| 2. Boolean encoding             | &#x20;Pass | The matrix correctly represents item presence using True/False values, not quantities.                                       |
| 3. Transaction retention        | &#x20;Pass | All 9,465 transactions, including 3,948 single-item baskets, are retained for the support denominator.                       |
| 4. Transformation verification  | &#x20;Pass | Transaction and item preservation, nonempty baskets, 18,887 item presences, and complete basket reconstruction are verified. |
| 5. Conclusions and pending work | &#x20;Pass | Successful transformation is distinguished from association-rule mining and evaluation, which remain pending.                |

### Verified results

The 9,465 × 94 Boolean matrix correctly represents the cleaned dataset. Its 18,887 True values match the number of unique transaction-item pairs.

Basket sizes range from 1 to 10 distinct items, averaging approximately 2.00. The complete reconstruction check confirms that each encoded basket preserves its cleaned item membership.

### Essential corrections

None identified.

### Final verdict

Phase 3 — Transformation is approved as written.

All five review criteria are satisfied. No essential code changes, revisions, or additional analyses are required.

Ready to proceed to Phase 4 — Data Mining.

### Phase 3 — Transformation — Revision 1

Round 1 identified no essential corrections. The transformation
code, results, and conclusions are retained unchanged.

The Boolean matrix represents 9,465 transactions and 94 items,
with 18,887 item presences. All transactions remain included,
including 3,948 single-item baskets.

Previously executed checks confirmed transaction and item
preservation, nonempty baskets, Boolean encoding, and exact
reconstruction of every cleaned basket.

No new execution was performed for this revision. Association-rule
mining and evaluation remain pending.

##Phase 3 — Transformation — Critique Round 2

## KDD Phase 3 — Transformation: Second Review (Revision 1)

Approved — No essential corrections

Overall assessment: Revision 1 remains adequately documented, methodologically sound, and consistent with the previously approved code and execution results.

| Review criterion                | Status     | Assessment                                                                                                                                       |
| ------------------------------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------------------ |
| 1. Basket construction          | &#x20;Pass | Transactions are correctly represented as baskets of distinct items.                                                                             |
| 2. Boolean encoding             | &#x20;Pass | The matrix represents item presence rather than quantity or purchase sequence.                                                                   |
| 3. Transaction retention        | &#x20;Pass | All 9,465 transactions, including 3,948 single-item baskets, remain included in the support denominator.                                         |
| 4. Transformation verification  | &#x20;Pass | Previously executed checks verified transaction and item preservation, nonempty baskets, 18,887 item presences, and exact basket reconstruction. |
| 5. Conclusions and pending work | &#x20;Pass | Completed transformation is clearly distinguished from pending association-rule mining and evaluation.                                           |

### Essential corrections

None identified.

Revision 1 accurately retains the previously verified findings without claiming additional execution.

### Final verdict

Phase 3 — Transformation is approved and complete.

All five review criteria remain satisfied. No essential issues, code changes, or additional analyses are required.

Ready to proceed to Phase 4 — Data Mining.

## Phase 3 — Transformation — Final Revision

Revision 1 is adopted following the second review. No essential
corrections or additional execution were required.

## Phase 4 — Data Mining — Initial Draft

### Mining approach
Use Apriori to identify frequent itemsets in the Boolean basket
matrix, then generate association rules.

All 9,465 historical transactions remain in the support denominator.
This is descriptive discovery using the available baskets, not
an independent evaluation of future purchase predictions.

### Settings fixed before execution
- Minimum support: 0.01.
- Minimum confidence: 0.30.
- Retain rules with lift greater than 1.
- Maximum itemset size: 3.
- Retain only rules with one consequent item.

These are project choices, not assignment-prescribed or optimized
thresholds. Support of 0.01 requires occurrence in at least 95
transactions, limiting patterns supported by very few baskets.
Confidence of 0.30 requires the consequent in at least 30% of
antecedent-containing baskets. Lift greater than 1 requires
co-occurrence above the independence expectation.

Limiting itemsets to three items keeps the rules interpretable.
A single consequent supports suggesting one item from one or
two existing basket items. Larger combinations are outside this
defined search.

### Metrics and ordering
For a rule A → B:
- Support is the fraction of all baskets containing A and B.
- Confidence is the fraction of baskets containing A that also
  contain B.
- Lift is confidence divided by the overall support of B.

Order retained rules by lift, confidence, and support descending,
then alphabetically by antecedent and consequent to resolve ties.

### Verification and limitations
Record the software version and check rule thresholds, finite
metrics, disjoint antecedents/consequents, and itemset sizes.

These rules describe historical associations. They do not establish
causation, purchase sequence, future accuracy, or increased sales.
Direct metric verification and interpretation follow in Phase 5.

In [7]:
%pip -q install mlxtend==0.23.4

In [8]:
import mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.30
MAX_ITEMSET_SIZE = 3

frequent_itemsets = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_SIZE
)

if frequent_itemsets.empty:
    raise ValueError("No frequent itemsets meet the fixed threshold.")

rules = association_rules(
    frequent_itemsets,
    num_itemsets=len(basket_matrix),
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

rules = rules.loc[
    rules["lift"].gt(1)
    & rules["consequents"].map(len).eq(1)
].copy()

rules["Antecedent"] = rules["antecedents"].map(
    lambda items: " + ".join(sorted(items))
)
rules["Consequent"] = rules["consequents"].map(
    lambda items: " + ".join(sorted(items))
)

rules = rules.sort_values(
    ["lift", "confidence", "support", "Antecedent", "Consequent"],
    ascending=[False, False, False, True, True]
).reset_index(drop=True)

assert frequent_itemsets["support"].ge(MIN_SUPPORT).all()
assert frequent_itemsets["itemsets"].map(len).le(MAX_ITEMSET_SIZE).all()

assert np.isfinite(
    rules[["support", "confidence", "lift"]].to_numpy()
).all()
assert rules["support"].between(MIN_SUPPORT, 1).all()
assert rules["confidence"].between(MIN_CONFIDENCE, 1).all()
assert rules["lift"].gt(1).all()

for rule in rules.itertuples():
    assert rule.antecedents
    assert len(rule.consequents) == 1
    assert rule.antecedents.isdisjoint(rule.consequents)
    assert len(rule.antecedents | rule.consequents) <= MAX_ITEMSET_SIZE

print("mlxtend version:", mlxtend.__version__)
print("Transactions used:", len(basket_matrix))
print("Minimum supporting baskets:",
      int(np.ceil(MIN_SUPPORT * len(basket_matrix))))
print("Frequent itemsets:", len(frequent_itemsets))
print("Retained rules:", len(rules))
print("Threshold, metric validity, and rule-structure checks passed.")

if rules.empty:
    print("No rules meet the fixed criteria; interpretation remains pending.")

display(
    rules[
        ["Antecedent", "Consequent", "support", "confidence", "lift"]
    ].head(10).round(4)
)

mlxtend version: 0.23.4
Transactions used: 9465
Minimum supporting baskets: 95
Frequent itemsets: 61
Retained rules: 14
Threshold, metric validity, and rule-structure checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Antecedent,Consequent,support,confidence,lift
0,Toast,Coffee,0.0237,0.7044,1.4724
1,Spanish Brunch,Coffee,0.0109,0.5988,1.2518
2,Medialuna,Coffee,0.0352,0.5692,1.1899
3,Pastry,Coffee,0.0475,0.5521,1.1542
4,Alfajores,Coffee,0.0197,0.5407,1.1302
5,Juice,Coffee,0.0206,0.5342,1.1167
6,Sandwich,Coffee,0.0382,0.5324,1.1128
7,Cake,Coffee,0.0547,0.5270,1.1015
8,Scone,Coffee,0.0181,0.5229,1.0931
9,Cookies,Coffee,0.0282,0.5184,1.0837


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

### Phase 4 — Data Mining — Results

Using mlxtend 0.23.4, Apriori identified 61 frequent itemsets
from 9,465 historical baskets. The fixed minimum support of
0.01 requires at least 95 supporting baskets.

After applying confidence ≥ 0.30, lift > 1, and a single-item
consequent requirement, 14 rules remained. Threshold, finite-metric,
and rule-structure checks passed.

Toast → Coffee ranks first under the predefined ordering:
- Support: approximately 2.37%.
- Confidence: approximately 70.44%.
- Lift: approximately 1.4724.

This means approximately 70.44% of historical baskets containing
Toast also contained Coffee. Coffee's occurrence among Toast
baskets was approximately 1.47 times its overall occurrence rate.

All ten displayed rules have Coffee as their consequent.
The preview shows ten of the fourteen retained rules.

These findings describe associations in the same historical
baskets used for mining. They do not establish purchase order,
causation, future recommendation accuracy, or sales improvements.

Direct count-based metric verification and final interpretation
remain pending for Phase 5.

##Phase 4 — Data Mining — Critique Round 1

## KDD Phase 4 — Data Mining: Expert Review

Approved — No essential corrections

Overall assessment: Phase 4 is methodologically sound, adequately documented, and supported by the provided code and execution results.

| Review criterion              | Status     | Assessment                                                                                                                                                                   |
| ----------------------------- | ---------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Mining approach            | &#x20;Pass | Apriori and association-rule generation appropriately support historical co-purchase discovery.                                                                              |
| 2. Mining parameters          | &#x20;Pass | Support ≥ 0.01, confidence ≥ 0.30, lift > 1, maximum itemset size of 3, and single-item consequents are clearly justified.                                                   |
| 3. Support denominator        | &#x20;Pass | All 9,465 baskets remain included. The minimum supporting count of 95 is correct.                                                                                            |
| 4. Software and validation    | &#x20;Pass | `mlxtend` 0.23.4 is recorded. Sorting, threshold checks, finite metrics, and rule-structure checks are implemented.                                                          |
| 5. Results and interpretation | &#x20;Pass | The reported 61 frequent itemsets, 14 retained rules, and Toast → Coffee interpretation are consistent with the output. No unsupported causal or predictive claims are made. |
| 6. Pending evaluation         | &#x20;Pass | Direct count-based metric verification and final interpretation are appropriately deferred to Phase 5.                                                                       |

### Verified results and limitations

The provided execution output supports the reported mining results.

Toast → Coffee has approximately 2.37% support, 70.44% confidence, and 1.4724 lift. Its interpretation correctly describes historical association rather than causation or future predictive performance.

The displayed `DeprecationWarning` originates from the Jupyter environment and does not indicate a failure of the mining procedure.

### Essential corrections

None identified.

### Final verdict

Phase 4 — Data Mining is approved as written.

All six review criteria are satisfied. No essential code changes, revisions, or additional analyses are required.

Ready to proceed to Phase 5 — Interpretation and Evaluation.

### Phase 4 — Data Mining — Revision 1

Round 1 identified no essential corrections. The mining code,
parameters, results, and conclusions are retained unchanged.

Apriori produced 61 frequent itemsets and 14 retained rules using
all 9,465 baskets. The predefined thresholds, itemset-size limit,
single-item consequent requirement, and ordering remain unchanged.

Toast → Coffee ranks first, with approximately 2.37% support,
70.44% confidence, and 1.4724 lift. These metrics describe
historical co-occurrence, not causation or future accuracy.

Previously executed threshold, metric-validity, and rule-structure
checks passed. No new execution was performed for this revision.

Direct count-based metric verification and final interpretation
remain pending for Phase 5.

##Phase 4 — Data Mining — Critique Round 2

## KDD Phase 4 — Data Mining: Second Review (Revision 1)

Approved — No essential corrections

Overall assessment: Revision 1 remains adequately documented, methodologically sound, and consistent with the previously approved code, parameters, and execution results.

| Review criterion              | Status     | Assessment                                                                                                                           |
| ----------------------------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------ |
| 1. Mining approach            | &#x20;Pass | Apriori and association-rule generation remain appropriate for historical co-purchase discovery.                                     |
| 2. Mining parameters          | &#x20;Pass | All five predefined restrictions remain justified and unchanged.                                                                     |
| 3. Support denominator        | &#x20;Pass | All 9,465 baskets remain included, with minimum support requiring 95 baskets.                                                        |
| 4. Software and validation    | &#x20;Pass | `mlxtend` 0.23.4, rule ordering, threshold checks, finite metrics, and rule-structure checks remain documented.                      |
| 5. Results and interpretation | &#x20;Pass | The 61 frequent itemsets, 14 retained rules, and Toast → Coffee metrics are accurately reported without causal or predictive claims. |
| 6. Pending evaluation         | &#x20;Pass | Direct count-based metric verification and final interpretation remain appropriately deferred to Phase 5.                            |

### Essential corrections

None identified.

Revision 1 accurately retains the previously verified findings without claiming additional execution.

### Final verdict

Phase 4 — Data Mining is approved and complete.

All six review criteria remain satisfied. No essential issues, code changes, or additional analyses are required.

Ready to proceed to Phase 5 — Interpretation and Evaluation.

## Phase 4 — Data Mining — Final Revision

Revision 1 is adopted following the second review. No essential
corrections or additional execution were required.

## Phase 5 — Interpretation and Evaluation — Initial Draft

### Verify rule metrics
Recalculate support, confidence, and lift for every retained rule
by counting item membership directly in baskets reconstructed
from the cleaned records.

For each rule A → B:
- Support = baskets containing both A and B / all baskets.
- Confidence = baskets containing both A and B / baskets containing A.
- Lift = confidence / the fraction of baskets containing B.

Compare these calculations with the mining output using numerical
tolerances of rtol=1e-9 and atol=1e-12.

### Interpret the findings
Inspect all retained rules and explain the leading rule using
actual basket counts. Compare its confidence with the consequent's
overall frequency to distinguish association from popularity.

The rules may identify candidates for basket-based suggestions.
They do not show that a suggestion would cause an additional purchase.

### Evaluate against the objective
The historical discovery objective is supported if the retained
rules meet the predefined criteria and their metrics agree with
direct basket counts.

This checks implementation and descriptive associations using
the same data used for mining. It is not an independent test of
recommendation accuracy or statistical significance.

### Limitations and next steps
Results depend on this historical sample and the chosen thresholds.
Quantity, purchase sequence, customer identity, and financial
benefits are not established. Timestamp accuracy and transaction
meaning have not been independently verified.

After evaluation, save the rules and demonstrate suggestions for
an example basket, excluding items already present. Artifact
packaging, the final report, Medium article, YouTube walkthrough,
and GitHub publication remain pending.

In [9]:
evaluation_baskets = (
    clean_df.groupby("TransactionNo")["Items"]
    .agg(lambda items: frozenset(items))
    .tolist()
)

n_baskets = len(evaluation_baskets)
assert n_baskets == len(basket_matrix)
assert not rules.empty, "No retained rules are available to evaluate."

verification_rows = []

for rule in rules.itertuples(index=False):
    antecedent = rule.antecedents
    consequent = rule.consequents

    n_a = sum(antecedent.issubset(b) for b in evaluation_baskets)
    n_b = sum(consequent.issubset(b) for b in evaluation_baskets)
    n_ab = sum(
        (antecedent | consequent).issubset(b)
        for b in evaluation_baskets
    )

    assert n_a > 0 and n_b > 0

    support = n_ab / n_baskets
    confidence = n_ab / n_a
    consequent_support = n_b / n_baskets
    lift = confidence / consequent_support

    matches = np.allclose(
        [support, confidence, lift],
        [rule.support, rule.confidence, rule.lift],
        rtol=1e-9,
        atol=1e-12
    )
    assert matches, f"Metric mismatch: {rule.Antecedent}"

    verification_rows.append({
        "Antecedent": rule.Antecedent,
        "Consequent": rule.Consequent,
        "Antecedent baskets": n_a,
        "Consequent baskets": n_b,
        "Joint baskets": n_ab,
        "Support": support,
        "Confidence": confidence,
        "Consequent baseline": consequent_support,
        "Lift": lift,
        "Metrics match": bool(matches)
    })

rule_verification = pd.DataFrame(verification_rows)

print("Rules verified:", len(rule_verification))
print("All count-based metric checks passed:",
      bool(rule_verification["Metrics match"].all()))

display(rule_verification.round(4))

leading = rule_verification.iloc[0]

print(
    f"\nLeading rule: {leading['Antecedent']} → "
    f"{leading['Consequent']}"
)
print(
    f"Support: {leading['Joint baskets']} / {n_baskets} "
    f"= {leading['Support']:.4%}"
)
print(
    f"Confidence: {leading['Joint baskets']} / "
    f"{leading['Antecedent baskets']} "
    f"= {leading['Confidence']:.4%}"
)
print(
    f"Consequent baseline: {leading['Consequent baskets']} / "
    f"{n_baskets} = {leading['Consequent baseline']:.4%}"
)
print(f"Lift: {leading['Lift']:.4f}")
print(
    "Confidence above consequent baseline (percentage points):",
    round(
        100 * (
            leading["Confidence"] - leading["Consequent baseline"]
        ),
        2
    )
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Rules verified: 14
All count-based metric checks passed: True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Antecedent,Consequent,Antecedent baskets,Consequent baskets,Joint baskets,Support,Confidence,Consequent baseline,Lift,Metrics match
0,Toast,Coffee,318,4528,224,0.0237,0.7044,0.4784,1.4724,True
1,Spanish Brunch,Coffee,172,4528,103,0.0109,0.5988,0.4784,1.2518,True
2,Medialuna,Coffee,585,4528,333,0.0352,0.5692,0.4784,1.1899,True
3,Pastry,Coffee,815,4528,450,0.0475,0.5521,0.4784,1.1542,True
4,Alfajores,Coffee,344,4528,186,0.0197,0.5407,0.4784,1.1302,True
5,Juice,Coffee,365,4528,195,0.0206,0.5342,0.4784,1.1167,True
6,Sandwich,Coffee,680,4528,362,0.0382,0.5324,0.4784,1.1128,True
7,Cake,Coffee,983,4528,518,0.0547,0.5270,0.4784,1.1015,True
8,Scone,Coffee,327,4528,171,0.0181,0.5229,0.4784,1.0931,True
9,Cookies,Coffee,515,4528,267,0.0282,0.5184,0.4784,1.0837,True



Leading rule: Toast → Coffee
Support: 224 / 9465 = 2.3666%
Confidence: 224 / 318 = 70.4403%
Consequent baseline: 4528 / 9465 = 47.8394%
Lift: 1.4724
Confidence above consequent baseline (percentage points): 22.6


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

### Phase 5 — Interpretation and Evaluation — Results

All 14 retained rules had support, confidence, and lift recalculated
directly from the cleaned baskets. Every comparison matched the
mining output within rtol=1e-9 and atol=1e-12.

### Leading rule: Toast → Coffee

Of 9,465 baskets:
- 318 contained Toast.
- 4,528 contained Coffee.
- 224 contained both.

Therefore:
- Support = 224 / 9,465 = approximately 2.37%.
- Confidence = 224 / 318 = approximately 70.44%.
- Coffee baseline = 4,528 / 9,465 = approximately 47.84%.
- Lift = approximately 1.4724.

Coffee appeared in Toast-containing baskets 22.60 percentage
points more often than in baskets overall. This is a historical
association, not an improvement caused by recommending Coffee.
The rule covers only 2.37% of all baskets despite its high confidence.

### Interpretation across retained rules

Thirteen of the fourteen rules suggest Coffee; the remaining rule
is Pastry → Bread. This concentration limits suggestion variety.

Pastry → Bread has approximately 33.87% confidence compared with
Bread's overall frequency of 32.72%, giving lift of approximately
1.0350. It passes the chosen criteria but shows only a small
difference from the baseline. Passing the thresholds does not
establish practical value or statistical significance.

### Evaluation conclusion

The historical discovery objective is supported: the procedure
identified rules meeting the fixed criteria, and direct basket
counts confirmed their metrics.

Evaluation used the same historical baskets as mining. Future
accuracy, customer response, and sales benefits remain unverified.
The results also depend on the chosen thresholds and the dataset's
limitations.

The next step is to save the rules and demonstrate basket-based
suggestions while excluding items already present. That demonstration
and the final submission materials remain pending.

##Phase 5 — Interpretation and Evaluation — Critique Round 1

## KDD Phase 5 — Interpretation and Evaluation: Expert Review

Approved — No essential corrections

Overall assessment: Phase 5 is methodologically sound, adequately documented, and supported by the provided code, findings, and execution output.

| Review criterion                  | Status     | Assessment                                                                                                                                                    |
| --------------------------------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Direct metric verification     | &#x20;Pass | All 14 rules had support, confidence, and lift independently recalculated from cleaned basket counts. All comparisons passed within the specified tolerances. |
| 2. Toast → Coffee interpretation  | &#x20;Pass | The counts, 2.3666% support, 70.4403% confidence, 47.8394% Coffee baseline, and 1.4724 lift are accurate. Limited joint support is acknowledged.              |
| 3. Rule diversity and modest lift | &#x20;Pass | The concentration of 13 Coffee consequents and the modest 1.0350 lift of Pastry → Bread are correctly interpreted.                                            |
| 4. Evaluation conclusion          | &#x20;Pass | Historical discovery is supported without claiming independent predictive validation, statistical significance, causation, or sales benefits.                 |
| 5. Limitations and pending tasks  | &#x20;Pass | Historical data limitations, unverified outcomes, saved-rule demonstration, and remaining deliverables are clearly documented.                                |

### Verified results and limitations

The executed results confirm that all 14 retained rules match direct basket-count calculations.

Toast → Coffee occurs in 224 of 9,465 baskets, with confidence approximately 22.60 percentage points above Coffee's overall historical frequency.

The interpretation correctly distinguishes historical association from demonstrated recommendation effectiveness.

The Jupyter `DeprecationWarning` does not indicate a failure in the evaluation code.

### Essential corrections

None identified.

### Final verdict

Phase 5 — Interpretation and Evaluation is approved as written.

All five review criteria are satisfied. No essential code changes, revisions, or additional analyses are required.

Phase 5 is complete. The project is ready for the saved-rule recommendation demonstration and final deliverables.

### Phase 5 — Interpretation and Evaluation — Revision 1

Round 1 identified no essential corrections. The evaluation code,
findings, and conclusions are retained unchanged.

Direct basket-count calculations matched support, confidence,
and lift for all 14 retained rules within the stated tolerances.

Toast → Coffee occurred in 224 of 9,465 baskets. Its confidence
was approximately 70.44%, compared with Coffee's overall frequency
of 47.84%, giving lift of approximately 1.4724.

The concentration of 13 Coffee consequents and the modest lift
of Pastry → Bread remain documented.

These findings support historical association discovery.
Independent predictive accuracy, statistical significance,
causal effects, and sales benefits remain unverified.

No new execution was performed for this revision. The saved-rule
suggestion demonstration and final submission materials remain pending.

## Phase 5 — Interpretation and Evaluation — Critique Round 2

## KDD Phase 5 — Interpretation and Evaluation: Second Review (Revision 1)

Approved — No essential corrections

Overall assessment: Revision 1 remains adequately documented, methodologically sound, and consistent with the previously approved code, execution results, and conclusions.

| Review criterion                  | Status     | Assessment                                                                                                                                        |
| --------------------------------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Direct metric verification     | &#x20;Pass | Support, confidence, and lift were verified for all 14 rules using direct basket counts within the specified tolerances.                          |
| 2. Toast → Coffee interpretation  | &#x20;Pass | The basket counts, 70.44% confidence, 47.84% Coffee baseline, 1.4724 lift, and limited support remain accurately documented in the approved work. |
| 3. Rule diversity and modest lift | &#x20;Pass | The concentration of 13 Coffee consequents and modest lift of Pastry → Bread are appropriately acknowledged.                                      |
| 4. Evaluation conclusion          | &#x20;Pass | Historical discovery is distinguished from unverified predictive accuracy, statistical significance, causation, and sales benefits.               |
| 5. Limitations and pending tasks  | &#x20;Pass | Limitations remain documented, and the pending demonstration and final deliverables are clearly identified.                                       |

### Essential corrections

None identified.

Revision 1 accurately retains the previously verified findings without claiming additional execution.

### Final verdict

Phase 5 — Interpretation and Evaluation is approved and complete.

All five review criteria remain satisfied. No essential issues, code changes, or additional analyses are required.

Ready to proceed with the saved-rule suggestion demonstration and final deliverables.

## Phase 5 — Interpretation and Evaluation — Final Revision

Revision 1 is adopted following the second review. No essential
corrections or additional evaluation were required.

## Saved-Rule Suggestion Demonstration

Save the retained rules, item vocabulary, and mining settings
as JSON, then reload them without rerunning Apriori.

The suggestion function accepts a list of item names, trims
surrounding whitespace, and rejects unknown or blank names.
Capitalization remains significant.

A rule applies when all antecedent items are present and its
consequent is absent. Matching rules follow the existing order:
lift, confidence, and support descending, then alphabetical
antecedent and consequent. If several rules suggest the same
item, retain its highest-ranked matching rule.

Return up to three distinct suggestions. Empty baskets or baskets
with no applicable rule return no suggestions.

Verify that saving and reloading preserves the rules and produces
identical suggestions. Check that suggested items are not already
present and that every suggestion has a matching antecedent.

This is a functional demonstration using example baskets, not
a test of future purchase accuracy or increased sales.

In [11]:
import json
import platform
from pathlib import Path

artifact_dir = Path("03_KDD/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

rule_records = [
    {
        "antecedent": sorted(row.antecedents),
        "consequent": sorted(row.consequents)[0],
        "support": float(row.support),
        "confidence": float(row.confidence),
        "lift": float(row.lift)
    }
    for row in rules.itertuples(index=False)
]

rule_bundle = {
    "items": sorted(basket_matrix.columns.tolist()),
    "settings": {
        "min_support": MIN_SUPPORT,
        "min_confidence": MIN_CONFIDENCE,
        "lift_greater_than": 1,
        "max_itemset_size": MAX_ITEMSET_SIZE,
        "single_item_consequent": True,
        "ordering": [
            "lift descending", "confidence descending",
            "support descending", "antecedent alphabetical",
            "consequent alphabetical"
        ]
    },
    "rules": rule_records
}

model_path = artifact_dir / "basket_rules.json"
model_path.write_text(
    json.dumps(rule_bundle, indent=2, allow_nan=False),
    encoding="utf-8"
)
loaded_bundle = json.loads(model_path.read_text(encoding="utf-8"))
assert loaded_bundle == rule_bundle


def suggest_items(items, bundle, top_n=3):
    if not isinstance(items, (list, tuple, set, frozenset)):
        raise TypeError("Provide a collection of item names.")
    if not all(isinstance(item, str) and item.strip() for item in items):
        raise ValueError("Item names must be nonblank strings.")
    if type(top_n) is not int or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    basket = {item.strip() for item in items}
    unknown = basket - set(bundle["items"])
    if unknown:
        raise ValueError(f"Unknown items: {sorted(unknown)}")

    suggestions = []
    already_suggested = set()

    for rule in bundle["rules"]:
        item = rule["consequent"]
        if (
            set(rule["antecedent"]).issubset(basket)
            and item not in basket
            and item not in already_suggested
        ):
            suggestions.append({
                "Suggested item": item,
                "Matching antecedent": " + ".join(rule["antecedent"]),
                "Support": rule["support"],
                "Confidence": rule["confidence"],
                "Lift": rule["lift"]
            })
            already_suggested.add(item)

        if len(suggestions) == top_n:
            break

    return suggestions


example_baskets = [
    ["Toast"],
    ["Pastry"],
    ["Toast", "Coffee"],
    []
]

demo_rows = []

for items in example_baskets:
    original = suggest_items(items, rule_bundle)
    reloaded = suggest_items(items, loaded_bundle)
    assert original == reloaded
    assert len({r["Suggested item"] for r in reloaded}) == len(reloaded)

    for suggestion in reloaded:
        assert suggestion["Suggested item"] not in items
        assert any(
            set(rule["antecedent"]).issubset(set(items))
            and rule["consequent"] == suggestion["Suggested item"]
            for rule in loaded_bundle["rules"]
        )

    demo_rows.append({
        "Input basket": json.dumps(items),
        "Suggestions": json.dumps(
            [r["Suggested item"] for r in reloaded]
        ),
        "Number of suggestions": len(reloaded),
        "Reproduced after reload": original == reloaded
    })

assert suggest_items([], loaded_bundle) == []

demo_results = pd.DataFrame(demo_rows)

print("Saved rules:", model_path)
print("Rules preserved after reload:", loaded_bundle == rule_bundle)
print("Saved rule count:", len(loaded_bundle["rules"]))
print("Suggestion reproduction and membership checks passed.")
display(demo_results)
display(pd.DataFrame(suggest_items(["Toast"], loaded_bundle)).round(4))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Saved rules: 03_KDD/artifacts/basket_rules.json
Rules preserved after reload: True
Saved rule count: 14
Suggestion reproduction and membership checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Input basket,Suggestions,Number of suggestions,Reproduced after reload
0,"[""Toast""]","[""Coffee""]",1,True
1,"[""Pastry""]","[""Coffee"", ""Bread""]",2,True
2,"[""Toast"", ""Coffee""]",[],0,True
3,[],[],0,True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Suggested item,Matching antecedent,Support,Confidence,Lift
0,Coffee,Toast,0.0237,0.7044,1.4724


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [12]:
rules_export = rules[
    ["Antecedent", "Consequent", "support", "confidence", "lift"]
].copy()

itemsets_export = frequent_itemsets.copy()
itemsets_export["itemsets"] = itemsets_export["itemsets"].map(
    lambda items: json.dumps(sorted(items))
)

exports = {
    "association_rules.csv": rules_export,
    "frequent_itemsets.csv": itemsets_export,
    "rule_verification.csv": rule_verification,
    "suggestion_demo.csv": demo_results
}

export_checks = []

for filename, frame in exports.items():
    path = artifact_dir / filename
    frame.to_csv(path, index=False)
    restored = pd.read_csv(path)

    pd.testing.assert_frame_equal(
        frame.reset_index(drop=True),
        restored,
        check_dtype=False,
        check_exact=False,
        rtol=1e-9,
        atol=1e-12
    )

    export_checks.append({
        "File": filename,
        "Rows": len(restored),
        "Matches notebook data": True
    })

schema = {
    "input": "Collection of item-name strings",
    "normalization": "Trim surrounding whitespace; preserve case",
    "allowed_items": loaded_bundle["items"],
    "unknown_or_blank_items": "Reject with an error",
    "empty_or_unmatched_basket": "Return an empty suggestion list",
    "default_max_suggestions": 3,
    "output": [
        "Suggested item", "Matching antecedent",
        "Support", "Confidence", "Lift"
    ]
}

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "mlxtend": mlxtend.__version__
}

for filename, content in {
    "input_schema.json": schema,
    "environment_versions.json": environment
}.items():
    path = artifact_dir / filename
    path.write_text(json.dumps(content, indent=2), encoding="utf-8")
    assert json.loads(path.read_text(encoding="utf-8")) == content

display(pd.DataFrame(export_checks))
print("Schema and environment JSON checks passed.")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,File,Rows,Matches notebook data
0,association_rules.csv,14,True
1,frequent_itemsets.csv,61,True
2,rule_verification.csv,14,True
3,suggestion_demo.csv,4,True


Schema and environment JSON checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Demonstration Results and Delivery Status

All 14 rules were saved to artifacts/basket_rules.json and
preserved exactly after reloading. Suggestions reproduced
identically for all four demonstration inputs.

Toast produced Coffee; Pastry produced Coffee and Bread.
Toast with Coffee already present produced no suggestions.
An empty basket also produced no suggestions.

The executed checks confirmed distinct suggestions, exclusion
of existing basket items, and matching rule antecedents.

All exported CSVs matched their notebook DataFrames:
- association_rules.csv: 14 rows.
- frequent_itemsets.csv: 61 rows.
- rule_verification.csv: 14 rows.
- suggestion_demo.csv: 4 rows.

Input-schema and environment JSON round-trip checks passed.

The saved JSON contains rules, vocabulary, and mining settings.
The notebook's suggest_items function performs input validation,
rule matching, and suggestion selection. Both are needed to
reproduce the demonstration without rerunning Apriori.

All five KDD phases and the saved-rule demonstration are complete.
The results establish historical associations and functional
suggestion behavior, not future accuracy or sales benefits.

Final report preparation, GitHub packaging/publication, the
Medium article, and the YouTube walkthrough remain pending.

In [13]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "03_KDD_artifacts",
    "zip",
    root_dir="03_KDD",
    base_dir="artifacts"
)
files.download(archive_path)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag